# Deterministic Horizon — 60-second quickstart

This notebook reproduces the headline plot of the paper using a **fully offline** synthetic reasoner. No API keys, no GPUs, ~10 seconds runtime.

We compare:
- **C1 — Neural CoT** (simulated with context-dependent error $\varepsilon(d) = \varepsilon_0 + \gamma d/L$ from Theorem 1)
- **C3 — Tool-integrated** (exact BFS solver)

and fit the super-exponential decay model to recover the **Deterministic Horizon $d^*$**.

> For the paper's full LLM evaluations across 12 models, see `examples/run_paper_experiments.py`.

In [ ]:
# 1) Load cached sample results (180 synthetic evaluations across depths 4-40)
import json, pathlib

results = json.loads(pathlib.Path('../results/sample/synthetic_results.json').read_text())
print(f'loaded {len(results)} results across conditions C1 (neural) and C3 (tool)')

In [ ]:
# 2) Estimate the Deterministic Horizon d*
from deterministic_horizon.metrics import estimate_horizon, accuracy_by_depth

cot = [r for r in results if r['condition'] == 'C1']
horizon = estimate_horizon(cot, threshold=0.5)
print(f"d* = {horizon['d_star']:.1f}  "
      f"(95% CI [{horizon.get('d_star_ci_low', float('nan')):.1f}, "
      f"{horizon.get('d_star_ci_high', float('nan')):.1f}])")
print(f"decoherence-model fit: R² = {horizon.get('r_squared', float('nan')):.3f}")

In [ ]:
# 3) Reproduce the hero figure
from deterministic_horizon.analysis import generate_figures

figs = generate_figures(results, output_dir='quickstart_out', title='Deterministic Horizon — quickstart')
from IPython.display import Image, display
for f in figs:
    display(Image(str(f)))

In [ ]:
# 4) Inspect the accuracy table
acc = accuracy_by_depth(cot)
print(f"{'depth':>5} {'accuracy':>10} {'95% CI':>20} {'N':>4}")
for d in sorted(acc):
    s = acc[d]
    print(f"{d:>5} {s['accuracy']:>9.1%} [{s['ci_low']:.2f}, {s['ci_high']:.2f}]   {s['n']:>4}")

## Next steps

- **Run on real models** — `dh evaluate --model gpt-4o --instances data/sample/permutation_n8.json --conditions C1,C3 --output results/gpt4o.json`
- **Try other tasks** — swap `PermutationTask` for `FSATask` or `ArithmeticTask`
- **Reproduce paper tables** — `make paper-tables` (runs every model × condition)
- **Read the paper** — [`paper/ICML2026_DeterministicHorizon_FINAL.pdf`](../paper/ICML2026_DeterministicHorizon_FINAL.pdf)